# Utils - QB - DurationConverter

Ce notebook illustre et vérifie le comportement de la classe `DurationConverter`
(`tsforecast/utils/duration/converter.py`), utilisée dans tout le package pour
convertir des durées entre unités (via `convert()`) et pour calculer des facteurs
de conversion entre fréquences/durées (via `get_conversion_factor()`) — ce dernier
point est notamment central au fonctionnement de l'imputation multi-fréquences
(`HighFrequencyImputer` / `FrequencyConverter`) et des délais de publication
(`PublicationDelayTransformer`).

**Périmètre** : seules les méthodes publiques de `DurationConverter` sont testées :
- `convert(value, from_unit, to_unit, rounding=None)`
- `get_conversion_factor(from_unit, to_unit)`

La méthode privée `_round_result()` n'est pas testée directement (elle est couverte
indirectement via le paramètre `rounding` de `convert()`).

**Remarque sur les jeux de données** : contrairement aux classes qui manipulent des
`Series`/`DataFrame` temporels, `DurationConverter` opère sur des valeurs scalaires
et des unités de durée (chaînes de caractères). Il n'est donc pas pertinent de lui
passer directement les jeux de données créés dans
`3 - QB - Panel a frequences mixtes heterogene.ipynb`. On réutilise en revanche, dans
la dernière section, les **fréquences et délais de publication** définis dans ce
notebook (PIB trimestriel avec 2 mois de délai, inflation mensuelle avec 1 mois de
délai, etc.) comme valeurs d'entrée réalistes.

Ce notebook a vocation à servir de base à de futurs tests unitaires
(`tests/utils/duration/test_converter.py`, dossier qui n'existe pas encore).

## 1 - Import et instanciation

In [ ]:
# Importation des modules
from typing import get_args

# Classe testée
from tsforecast.utils.duration.converter import DurationConverter

# Types exportés (pour lister les unités supportées, à titre indicatif)
from tsforecast.utils.duration.normalizer import DurationType, UserDurationType

# Instanciation du convertisseur
converter = DurationConverter()

print("Unités (codes) déclarées dans le type DurationType :")
print(get_args(DurationType))
print()
print("Unités (littéraux) déclarées dans le type UserDurationType :")
print(get_args(UserDurationType))

## 2 - `convert()` : comportement de base

### 2.1 - Conversions simples entre codes

In [ ]:
# Quelques conversions simples entre codes de durée
print(converter.convert(2.5, 'h', 'min'))    # 2.5 heures -> minutes
print(converter.convert(90, 'min', 'h'))     # 90 minutes -> heures
print(converter.convert(1.5, 'D', 'h'))      # 1.5 jour -> heures
print(converter.convert(1, 'W', 'D'))        # 1 semaine -> jours

### 2.2 - Formats d'entrée mixtes (codes vs littéraux)

In [ ]:
# Le normalizer interne accepte indifféremment codes et noms littéraux,
# et peut mélanger les deux formats entre from_unit et to_unit
resultats = {
    "code -> code": converter.convert(1, 'D', 'h'),
    "littéral -> littéral": converter.convert(1, 'day', 'hour'),
    "code -> littéral": converter.convert(1, 'D', 'hour'),
    "littéral -> code": converter.convert(1, 'day', 'h'),
}
for label, valeur in resultats.items():
    print(f"{label:25s} -> {valeur}")

assert len(set(resultats.values())) == 1, "Les 4 formats devraient donner le même résultat"

### 2.3 - Conversion identité et matrice complète

In [ ]:
import pandas as pd

unites = ['ns', 'us', 'ms', 's', 'min', 'h', 'D', 'B', 'W', 'SM', 'M', 'Q', 'Y']

# Vérification : convertir une unité vers elle-même renvoie toujours la valeur inchangée
for u in unites:
    assert converter.convert(5, u, u) == 5, f"Identité en échec pour {u}"
print("OK : convert(x, u, u) == x pour toutes les unités testées")

# Matrice complète des facteurs convert(1, from, to) -> utile comme table de référence
matrice = pd.DataFrame(
    {to: [converter.convert(1, frm, to) for frm in unites] for to in unites},
    index=unites,
)
matrice.index.name = "from \\ to"
matrice

### 2.4 - Aller-retour (round-trip)

In [ ]:
import itertools
import numpy as np

# Vérification que convert(convert(v, a, b), b, a) redonne v (aux erreurs de flottants près)
valeurs_test = [0.0, 1.0, 3.7, 1000.0]
for v, (a, b) in itertools.product(valeurs_test, itertools.combinations(unites, 2)):
    aller = converter.convert(v, a, b)
    retour = converter.convert(aller, b, a)
    assert np.isclose(retour, v), f"Round-trip en échec pour {a}->{b}->{a} avec v={v}"
print("OK : round-trip convert(convert(v, a, b), b, a) == v pour toutes les paires et valeurs testées")

### 2.5 - Valeurs limites : zéro, négatif, très grand / très petit

In [ ]:
# Zéro
print("zéro           :", converter.convert(0, 'D', 'h'))

# Valeur négative : AUCUNE validation de signe n'est effectuée par convert()
# (une durée négative n'a pas de sens métier mais n'est pas rejetée)
print("valeur négative:", converter.convert(-5, 'h', 'min'))

# Valeurs extrêmes
print("très grand     :", converter.convert(1e12, 'ns', 'Y'))
print("très petit     :", converter.convert(1e-6, 'Y', 'ns'))

### 2.6 - Erreurs : unités inconnues et types invalides

In [ ]:
# Unité inconnue (from_unit ou to_unit) -> ValueError, levée par le normalizer interne
for from_u, to_u in [('foo', 'D'), ('D', 'foo')]:
    try:
        converter.convert(1, from_u, to_u)
        print(f"convert(1, {from_u!r}, {to_u!r}) : PAS D'ERREUR (inattendu)")
    except ValueError as e:
        print(f"convert(1, {from_u!r}, {to_u!r}) -> ValueError : {e}")

# Valeur non numérique -> TypeError, car convert() fait simplement `value * facteur`
for valeur_invalide in ['abc', None, [1, 2, 3]]:
    try:
        converter.convert(valeur_invalide, 'D', 'h')
        print(f"convert({valeur_invalide!r}, ...) : PAS D'ERREUR (inattendu)")
    except TypeError as e:
        print(f"convert({valeur_invalide!r}, 'D', 'h') -> TypeError : {e}")

## 3 - Paramètre `rounding`

### 3.1 - `None` vs `'floor'` vs `'ceil'`

In [ ]:
heures = 37  # 37 heures = 1.5417 jours (valeur non entière)

exact = converter.convert(heures, 'h', 'D', rounding=None)
floor = converter.convert(heures, 'h', 'D', rounding='floor')
ceil_ = converter.convert(heures, 'h', 'D', rounding='ceil')

print(f"rounding=None  : {exact!r} ({type(exact).__name__})")
print(f"rounding=floor : {floor!r} ({type(floor).__name__})")
print(f"rounding=ceil  : {ceil_!r} ({type(ceil_).__name__})")

### 3.2 - Le type de retour change selon `rounding`

In [ ]:
# Point important : rounding=None renvoie toujours un float, alors que
# rounding='floor'/'ceil' renvoient un int (via math.floor / math.ceil),
# même quand la valeur d'entrée était déjà un int et le résultat un nombre entier.
print(type(converter.convert(2, 'D', 'h', rounding=None)))    # float
print(type(converter.convert(2, 'D', 'h', rounding='floor'))) # int
print(type(converter.convert(2, 'D', 'h', rounding='ceil')))  # int

### 3.3 - `rounding` avec des valeurs négatives

In [ ]:
# floor / ceil suivent la convention mathématique standard, y compris pour les négatifs
valeur = -37  # heures
print("floor(-37h en D) :", converter.convert(valeur, 'h', 'D', rounding='floor'))  # -2 (arrondi vers -inf)
print("ceil(-37h en D)  :", converter.convert(valeur, 'h', 'D', rounding='ceil'))   # -1 (arrondi vers +inf)

### 3.4 - Valeur de `rounding` invalide : comportement silencieux (point de vigilance)

In [ ]:
# ATTENTION : passer une valeur de rounding non reconnue (autre que 'floor'/'ceil'/None)
# NE LÈVE PAS D'ERREUR. `_round_result` retourne alors la valeur non arrondie,
# silencieusement. C'est un comportement à documenter/tester explicitement, et
# potentiellement à durcir (lever une ValueError) dans une future révision du code.
resultat_valide = converter.convert(37, 'h', 'D', rounding=None)
resultat_invalide = converter.convert(37, 'h', 'D', rounding='round')  # 'round' n'existe pas

print("rounding=None    :", resultat_valide)
print("rounding='round' :", resultat_invalide, "(identique à rounding=None, aucune erreur levée)")
assert resultat_valide == resultat_invalide

## 4 - `get_conversion_factor()`

### 4.1 - Facteurs de base et cohérence avec `convert()`

In [ ]:
# Quelques facteurs de référence
print("h -> min :", converter.get_conversion_factor('h', 'min'))
print("D -> h   :", converter.get_conversion_factor('D', 'h'))
print("W -> D   :", converter.get_conversion_factor('W', 'D'))

# Cohérence : convert(v, a, b) == v * get_conversion_factor(a, b)
for v, a, b in [(3, 'h', 'min'), (2, 'D', 'h'), (1.5, 'W', 'D')]:
    assert converter.convert(v, a, b) == v * converter.get_conversion_factor(a, b)
print("OK : convert(v, a, b) == v * get_conversion_factor(a, b)")

### 4.2 - Symétrie : `factor(a, b) * factor(b, a) == 1`

In [ ]:
for a, b in itertools.combinations(unites, 2):
    fab = converter.get_conversion_factor(a, b)
    fba = converter.get_conversion_factor(b, a)
    assert np.isclose(fab * fba, 1.0), f"Symétrie en échec pour {a}/{b}"
print("OK : get_conversion_factor(a, b) * get_conversion_factor(b, a) == 1 pour toutes les paires")

### 4.3 - Approximations calendaires : `SM`, `M`, `Q`, `Y` ne sont pas des multiples exacts les uns des autres

In [ ]:
# D et W sont des durées exactes (respectivement 86400s et 604800s).
# En revanche SM (15j), M (30j), Q (90j) et Y (365j) sont des APPROXIMATIONS
# calendaires. Elles ne sont donc pas toujours des multiples entiers exacts
# les unes des autres, ce qui peut surprendre.
paires_calendaires = [
    ('Q', 'M', 3),     # 1 trimestre = 3 mois : exact ici
    ('Y', 'M', 12),    # 1 année = 12 mois : PAS exact (365/30)
    ('Y', 'Q', 4),     # 1 année = 4 trimestres : PAS exact (365/90)
    ('M', 'W', None),  # 1 mois en semaines : pas de valeur ronde attendue
    ('Y', 'W', None),  # 1 année en semaines : pas de valeur ronde attendue
]

for frm, to, attendu in paires_calendaires:
    facteur = converter.get_conversion_factor(frm, to)
    if attendu is None:
        print(f"{frm} -> {to} : {facteur:.4f} (pas de valeur entière attendue)")
    else:
        ecart = facteur - attendu
        print(f"{frm} -> {to} : {facteur:.4f} (attendu ~{attendu}, écart de {ecart:+.4f})")

### 4.4 - `'B'` (jour ouvré) est traité comme `'D'` (jour calendaire)

In [ ]:
# Limitation à connaître : le facteur de conversion pour 'B' (business_day) est
# identique à celui de 'D' (day). DurationConverter ne tient donc PAS compte du fait
# qu'une semaine ne contient que 5 jours ouvrés sur 7 jours calendaires.
print("B -> D :", converter.get_conversion_factor('B', 'D'))  # 1.0, pas 5/7
print("B -> h :", converter.get_conversion_factor('B', 'h'))  # identique à D -> h

### 4.5 - Erreurs : unités non supportées

In [ ]:
for frm, to in [('foo', 'D'), ('D', 'foo')]:
    try:
        converter.get_conversion_factor(frm, to)
        print(f"get_conversion_factor({frm!r}, {to!r}) : PAS D'ERREUR (inattendu)")
    except ValueError as e:
        print(f"get_conversion_factor({frm!r}, {to!r}) -> ValueError : {e}")

## 5 - Application aux fréquences et délais du notebook `3 - QB - Panel a frequences mixtes heterogene`

Les valeurs ci-dessous reprennent les fréquences de publication et délais définis dans
`3 - QB - Panel a frequences mixtes heterogene.ipynb` (section 2) : PIB trimestriel avec
2 mois de délai, inflation/chômage mensuels avec 1 mois de délai, balance commerciale
annuelle avec 3 mois de délai, et dépenses publiques publiées annuellement (France,
Italie) ou trimestriellement (Allemagne) avec un délai d'un trimestre.

In [ ]:
indicateurs = {
    "pib_trimestriel": {"frequence": "Q", "delai": (2, "M")},
    "inflation_ipc": {"frequence": "M", "delai": (1, "M")},
    "taux_chomage": {"frequence": "M", "delai": (1, "M")},
    "balance_commerciale_annuelle": {"frequence": "Y", "delai": (3, "M")},
    "depenses_publiques_pib (FR/IT)": {"frequence": "Y", "delai": (1, "Q")},
    "depenses_publiques_pib (DE)": {"frequence": "Q", "delai": (1, "Q")},
}

# Pour chaque indicateur, on exprime le délai de publication en jours (unité "physique"
# commune) ainsi qu'en nombre de périodes de sa propre fréquence de publication.
for nom, infos in indicateurs.items():
    valeur_delai, unite_delai = infos["delai"]
    freq = infos["frequence"]
    delai_en_jours = converter.convert(valeur_delai, unite_delai, 'D')
    delai_en_periodes_freq = converter.convert(valeur_delai, unite_delai, freq)
    print(
        f"{nom:35s} freq={freq:2s} delai={valeur_delai} {unite_delai:2s} "
        f"-> {delai_en_jours:6.2f} jours (~{delai_en_periodes_freq:.2f} période(s) de {freq})"
    )

### 5.1 - Reproduction du calcul utilisé dans `PublicationDelayTransformer`

In [ ]:
# Reproduction du calcul effectué dans
# PublicationDelayTransformer._convert_shift_periods_to_index_periods
# (tsforecast/delays/transformers.py) : convertir un nombre de périodes exprimées
# dans la fréquence de la variable ("freq_source") en nombre de périodes équivalent
# dans la fréquence de l'index cible ("freq_cible"), avec arrondi au plus proche.
def n_periodes_equivalentes(n_periodes: float, freq_source: str, freq_cible: str) -> int:
    facteur = converter.get_conversion_factor(freq_source, freq_cible)
    return round(n_periodes * facteur)

# PIB : délai de 2 mois, index mensuel des séries temporelles -> 2 périodes mensuelles
print("PIB (délai 2 M, index M)     :", n_periodes_equivalentes(2, 'M', 'M'), "périodes")
# PIB : même délai de 2 mois, mais exprimé sur un index journalier
print("PIB (délai 2 M, index D)     :", n_periodes_equivalentes(2, 'M', 'D'), "périodes")
# Balance commerciale : délai de 3 mois, index mensuel
print("Balance (délai 3 M, index M) :", n_periodes_equivalentes(3, 'M', 'M'), "périodes")
# Dépenses publiques Allemagne : délai d'1 trimestre, index mensuel
print("Dépenses DE (délai 1 Q, index M) :", n_periodes_equivalentes(1, 'Q', 'M'), "périodes")

## 6 - Synthèse : pistes pour de futurs tests unitaires

Comportements observés dans ce notebook, à couvrir explicitement dans
`tests/utils/duration/test_converter.py` :

- **Identité** : `convert(x, u, u) == x` pour toutes les unités supportées.
- **Round-trip** : `convert(convert(v, a, b), b, a) == v` (aux erreurs de flottants près).
- **Formats mixtes** : `convert()` accepte indifféremment codes et noms littéraux, et peut
  mélanger les deux formats entre `from_unit` et `to_unit`.
- **Cohérence** : `convert(v, a, b) == v * get_conversion_factor(a, b)`.
- **Symétrie** : `get_conversion_factor(a, b) * get_conversion_factor(b, a) == 1`.
- **Type de retour de `convert()`** : `float` si `rounding=None`, `int` si
  `rounding='floor'` ou `'ceil'` — y compris quand le résultat exact est déjà un entier.
- **`rounding` invalide** : une valeur non reconnue (ex. `'round'`) n'est PAS rejetée ;
  `_round_result()` retourne silencieusement la valeur non arrondie. À couvrir par un
  test de régression, et à considérer comme un point à durcir (lever `ValueError`) côté
  code.
- **Pas de validation de signe** : les valeurs négatives sont acceptées sans erreur, alors
  qu'une durée négative n'a pas de sens métier.
- **Unités non supportées** : `ValueError`, pour `convert()` comme pour
  `get_conversion_factor()`, que l'unité invalide soit `from_unit` ou `to_unit`.
- **Types non numériques** : `TypeError` (propagée par `value * facteur`), pas de message
  d'erreur dédié au niveau de `DurationConverter`.
- **Approximations calendaires** : `SM`/`M`/`Q`/`Y` reposent sur des durées moyennes
  (15/30/90/365 jours) et ne sont pas des multiples entiers exacts les unes des autres
  (ex. `Y -> M` ≈ 12.1667, pas 12). Le code appelant qui a besoin d'un nombre entier de
  périodes doit explicitement arrondir (voir `FrequencyConverter`,
  `PublicationDelayTransformer`).
- **`'B'` (jour ouvré) == `'D'` (jour calendaire)** : le facteur de conversion ne tient pas
  compte des jours non ouvrés — à garder en tête si des délais sont un jour exprimés en
  jours ouvrés.